# Krishi Sahayak — Colab Pipeline

End-to-end pipeline: install deps -> build RAG index -> generate synthetic training data -> QLoRA fine-tune -> run inference -> evaluate.

**Runtime:** Go to `Runtime > Change runtime type` and select a **T4 GPU** (free tier works).

In [ ]:
!git clone https://github.com/YOUR_USERNAME/krishi-sahayak.git
%cd krishi-sahayak
!pip install -q -r requirements.txt

## 1. Build the RAG index

In [ ]:
%cd rag
!python build_index.py --schemes ../data/schemes.json --persist_dir ./chroma_store
%cd ..

## 2. Generate synthetic fine-tuning data

In [ ]:
%cd data
!python generate_synthetic_data.py --schemes schemes.json --out train_data.jsonl --n 400
%cd ..

## 3. QLoRA fine-tune (uses a T4-friendly 3B model)

In [ ]:
%cd finetune
!python train_qlora.py --train_file ../data/train_data.jsonl --output_dir ./krishi-sahayak-lora --epochs 3
%cd ..

## 4. Try it out

In [ ]:
%cd finetune
!python inference.py --query "I own 2 acres in Punjab and grow wheat, is there crop insurance for me?" --adapter_dir ./krishi-sahayak-lora --persist_dir ../rag/chroma_store
%cd ..

## 5. Evaluate: base vs RAG vs RAG+fine-tuned

In [ ]:
%cd eval
!python evaluate.py --config base
!python evaluate.py --config rag
!python evaluate.py --config rag_finetuned
%cd ..

## 6. Launch the demo app

In [ ]:
%cd app
!python app.py